In [41]:
# Interactive plotting

%reset -f
%matplotlib qt


import os
import easygui
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io

# Plot deformation

plt.close("all")

v_i = 0
v_f = 6000

# Open file dialog to select a .mat file
pitch_angle_file = "pitch_bladeA_messung24.mat"  # easygui.fileopenbox("Select the .mat file")

file_path_wo_pitch_korr = easygui.fileopenbox("Select the .csv file without correction")
file_path_w_pitch_korr = easygui.fileopenbox("Select the .csv file with correction")
fig = plt.figure()
ax = fig.add_subplot(111)

pitch_angle_data = scipy.io.loadmat(pitch_angle_file)
pitch_angle_data = pitch_angle_data["pitch_wea_bladeA"].flatten()

schwenk_data = np.loadtxt("deformation_V_6k.csv")
#pitch_angle_data = pitch_angle_data[v_i:v_f] - pitch_angle_data[0] + 90

path_list = [file_path_wo_pitch_korr, file_path_w_pitch_korr]
ax.plot(-schwenk_data, color="yellow")
k = 0
for path in path_list[:]:
    filename = os.path.basename(path)
    df_w_pitch = pd.read_csv(path, sep=";", decimal=',', skiprows=1, header=0)

    y_pos = df_w_pitch["Y [mm]"].to_numpy(copy=True)[v_i:v_f]
    sigma = df_w_pitch["sigma [pixel]"].to_numpy(copy=True)[v_i:v_f]
    deformation_V = df_w_pitch["V [mm]"].to_numpy(copy=True)[v_i:v_f]

    bad_sigmas = np.nonzero(sigma < 0)[0]
    deformation_V[bad_sigmas] = 0

    if k == 0:
        pass  # deformation_V = -1*deformation_V

    ax.plot(deformation_V, label=filename)

    k += 1

ax.set_title(f"Data from {filename}")

ax.set_ylabel("In-Plane Deformation [mm]")
ax.grid(True)
ax.set_xlim(v_i, v_f)
ax.legend(["Blender Baseline", "w/o Pitch Korrektur", "w/ Pitch Korrektur"])
plt.tight_layout()
plt.show()

# Create a plot for the variables found
fig1 = plt.figure()
ax1 = fig1.add_subplot()

ax1.plot(-pitch_angle_data)

ax1.set_xlabel("Index / Time")
ax1.set_ylabel(r"$\Delta \theta = \theta_{i} - \theta_{0}$ [°]")
ax1.grid(True)
#ax1.set_xlim(v_i, v_f)
plt.tight_layout()
plt.show()

TypeError: expected str, bytes or os.PathLike object, not NoneType